In [ ]:
# Solución de la Ecuación de Poisson 2D por Diferencias Finitas

Buscamos resolver la ecuación diferencial parcial elíptica de Poisson en un dominio rectangular $\Omega = [x_a, x_b] \times [y_a, y_b]$:

$$-\nabla^2 u(x,y) = -\left( \frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2} \right) = f(x,y) \quad \text{en } \Omega$$

Sujeta a condiciones de frontera tipo **Dirichlet**:
$$u(x,y) = g(x,y) \quad \text{en } \partial\Omega$$

### Discretización
Dividimos el dominio en una malla uniforme con pasos $h_x = \frac{x_b - x_a}{N_x}$ y $h_y = \frac{y_b - y_a}{N_y}$. Utilizando diferencias finitas centradas de segundo orden (plantilla de 5 puntos), obtenemos para cada nodo interior $(i,j)$:

$$-\left( \frac{u_{i+1,j} - 2u_{i,j} + u_{i-1,j}}{h_x^2} + \frac{u_{i,j+1} - 2u_{i,j} + u_{i,j-1}}{h_y^2} \right) = f(x_i, y_j)$$

Para un problema de prueba con solución analítica conocida:
* **Dominio:** $[0, 1] \times [0, 1]$
* **Solución exacta:** $u(x,y) = \sin(\pi x)\sin(\pi y)$
* **Término fuente:** $f(x,y) = 2\pi^2 \sin(\pi x)\sin(\pi y)$
* **Condición de frontera:** $u = 0$ en $\partial\Omega$

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla

# Configuración de gráficos
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 11

In [ ]:
# Límites del dominio rectangular [xa, xb] x [ya, yb]
xa, xb = 0.0, 1.0
ya, yb = 0.0, 1.0

# Número de subintervalos en cada dirección
Nx = 50
Ny = 50

# Tamaños de paso
hx = (xb - xa) / Nx
hy = (yb - ya) / Ny

# Generación de la malla
x = np.linspace(xa, xb, Nx + 1)
y = np.linspace(ya, yb, Ny + 1)
X, Y = np.meshgrid(x, y, indexing="ij")  # X[i,j], Y[i,j]


# Funciones del problema
def f_source(x, y):
  """Término fuente f(x,y) = -Laplaciano(u)"""
  return 2 * (np.pi**2) * np.sin(np.pi * x) * np.sin(np.pi * y)


def g_boundary(x, y):
  """Condición de frontera Dirichlet"""
  return np.sin(np.pi * x) * np.sin(np.pi * y)


def u_exact(x, y):
  """Solución analítica para validación"""
  return np.sin(np.pi * x) * np.sin(np.pi * y)

In [ ]:
def solve_poisson_2d(xa, xb, ya, yb, Nx, Ny, f_func, g_func):
  hx = (xb - xa) / Nx
  hy = (yb - ya) / Ny

  x = np.linspace(xa, xb, Nx + 1)
  y = np.linspace(ya, yb, Ny + 1)
  X, Y = np.meshgrid(x, y, indexing="ij")

  # Número de incógnitas interiores en cada eje
  nx_in = Nx - 1
  ny_in = Ny - 1
  N_total = nx_in * ny_in

  # Matriz solución inicializada con las condiciones de frontera
  U = np.zeros((Nx + 1, Ny + 1))
  U[0, :] = g_func(x[0], y)  # Frontera izquierda
  U[-1, :] = g_func(x[-1], y)  # Frontera derecha
  U[:, 0] = g_func(x, y[0])  # Frontera inferior
  U[:, -1] = g_func(x, y[-1])  # Frontera superior

  # Construcción de operadores 1D dispersos (segunda derivada negativa)
  # Dxx ~ (1/hx^2) * tridiag(-1, 2, -1)
  diag_x = 2.0 * np.ones(nx_in) / (hx**2)
  off_x = -1.0 * np.ones(nx_in - 1) / (hx**2)
  Dxx = sp.diags([off_x, diag_x, off_x], [-1, 0, 1], format="csc")

  # Dyy ~ (1/hy^2) * tridiag(-1, 2, -1)
  diag_y = 2.0 * np.ones(ny_in) / (hy**2)
  off_y = -1.0 * np.ones(ny_in - 1) / (hy**2)
  Dyy = sp.diags([off_y, diag_y, off_y], [-1, 0, 1], format="csc")

  # Producto de Kronecker para el operador Laplaciano 2D: A = Dxx (x) Iy + Ix (x) Dyy
  Ix = sp.eye(nx_in, format="csc")
  Iy = sp.eye(ny_in, format="csc")
  A = sp.kron(Dxx, Iy) + sp.kron(Ix, Dyy)

  # Construcción del vector del lado derecho (RHS) en nodos interiores
  RHS = f_func(X[1:-1, 1:-1], Y[1:-1, 1:-1]).copy()

  # Incorporar contribuciones de las fronteras Dirichlet al RHS
  RHS[0, :] += U[0, 1:-1] / (hx**2)
  RHS[-1, :] += U[-1, 1:-1] / (hx**2)
  RHS[:, 0] += U[1:-1, 0] / (hy**2)
  RHS[:, -1] += U[1:-1, -1] / (hy**2)

  # Aplanar RHS en orden fila-mayor (compatible con kron(Dxx, Iy))
  b = RHS.ravel()

  # Resolver el sistema lineal disperso A * u_in = b
  u_in = spla.spsolve(A, b)

  # Insertar solución interior en la malla completa
  U[1:-1, 1:-1] = u_in.reshape((nx_in, ny_in))

  return X, Y, U


# Ejecutar el solucionador
X, Y, U_num = solve_poisson_2d(xa, xb, ya, yb, Nx, Ny, f_source, g_boundary)
U_ex = u_exact(X, Y)

# Cálculo del error máximo (norma infinito)
error_max = np.max(np.abs(U_num - U_ex))
print(f"Error máximo absoluto (L_inf): {error_max:.4e}")

In [ ]:
fig = plt.figure(figsize=(14, 6))

# Gráfico 1: Superficie 3D de la solución numérica
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
surf = ax1.plot_surface(
    X, Y, U_num, cmap='viridis', edgecolor='none', alpha=0.9
)
ax1.set_title('Solución Numérica $u(x,y)$')
ax1.set_xlabel('$x$')
ax1.set_ylabel('$y$')
ax1.set_zlabel('$u$')
fig.colorbar(surf, ax=ax1, shrink=0.5, aspect=10)

# Gráfico 2: Error puntual absoluto |U_num - U_exact|
ax2 = fig.add_subplot(1, 2, 2)
error_map = np.abs(U_num - U_ex)
contour = ax2.contourf(X, Y, error_map, levels=30, cmap='magma')
ax2.set_title(f'Error Absoluto $|u_{{num}} - u_{{exact}}|$ (Max: {error_max:.2e})')
ax2.set_xlabel('$x$')
ax2.set_ylabel('$y$')
fig.colorbar(contour, ax=ax2)

plt.tight_layout()
plt.show()